In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torchvision.models as models
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tqdm import tqdm

# --- CONFIGURATION ---
MODEL_WEIGHTS_PATH = "/kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth"
CELEB_DF_PATH = "/kaggle/input/datasets/amanrawat001/celeb-df-preprocessed/Celeb-DF Preprocessed"

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. ARCHITECTURE ---
class SRMConv2d(nn.Module):
    def __init__(self):
        super().__init__()
        srm_kernel = torch.tensor([
            [[-1,  2, -1],
             [ 2, -4,  2],
             [-1,  2, -1]]
        ], dtype=torch.float32).unsqueeze(1) / 4.0
        self.weight = nn.Parameter(srm_kernel.repeat(3, 1, 1, 1), requires_grad=False)
        
    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, stride=1, padding=1, groups=3)

class DualStreamDeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.rgb_stream = models.efficientnet_b4(weights=None)
        rgb_feature_dim = self.rgb_stream.classifier[1].in_features
        self.rgb_stream.classifier = nn.Identity()
        
        self.srm_filter = SRMConv2d()
        self.freq_stream = models.resnet18(weights=None)
        self.freq_stream.fc = nn.Identity()
        freq_feature_dim = 512
        
        self.fusion = nn.Sequential(
            nn.Linear(rgb_feature_dim + freq_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
        
    def forward(self, x):
        spatial_features = self.rgb_stream(x)
        noise_x = self.srm_filter(x)
        freq_features = self.freq_stream(noise_x)
        combined = torch.cat((spatial_features, freq_features), dim=1)
        return self.fusion(combined)

# --- 2. LOAD & LOCK WEIGHTS ---
print("Initializing model architecture...")
model = DualStreamDeepfakeDetector()

print(f"Loading trained weights from: {MODEL_WEIGHTS_PATH}")
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

# --- 3. DATA LOADER ---
class CelebDFEvaluationDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        valid_extensions = {".jpg", ".jpeg", ".png"}
        all_images = [p for p in self.root_dir.rglob("*") if p.suffix.lower() in valid_extensions]
        
        for img_path in all_images:
            path_str = str(img_path).lower()
            
            if "real" in path_str:
                self.labels.append(0)
                self.image_paths.append(img_path)
            elif "fake" in path_str or "synthesis" in path_str:
                self.labels.append(1)
                self.image_paths.append(img_path)
                
        print(f"Benchmark Pool Ready: {len(self.image_paths)} total images loaded.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        label = self.labels[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, label

eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

celeb_dataset = CelebDFEvaluationDataset(root_dir=CELEB_DF_PATH, transform=eval_transform)
celeb_loader = DataLoader(celeb_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- 4. INFERENCE LOOP ---
print("\nStarting Cross-Dataset Evaluation on Celeb-DF...")
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(celeb_loader, desc="Running Inference"):
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu().numpy()
        
        all_preds.extend(probs)
        all_labels.extend(labels.numpy())

# --- 5. CALCULATE METRICS ---
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

binary_preds = (all_preds > 0.5).astype(int)

accuracy = accuracy_score(all_labels, binary_preds)
auc_roc = roc_auc_score(all_labels, all_preds)

print("\n" + "="*40)
print("FINAL CELEB-DF BENCHMARK RESULTS")
print("="*40)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUC-ROC:  {auc_roc:.4f}")
print("="*40)
print("\nDetailed Classification Report:")
print(classification_report(all_labels, binary_preds, target_names=["Real (0)", "Fake (1)"]))

Initializing model architecture...
Loading trained weights from: /kaggle/input/datasets/nithishkannaa/nonutdovemberlovember/best_dual_stream_model.pth
Benchmark Pool Ready: 65381 total images loaded.

Starting Cross-Dataset Evaluation on Celeb-DF...


Running Inference: 100%|██████████| 1022/1022 [06:46<00:00,  2.51it/s]


FINAL CELEB-DF BENCHMARK RESULTS
Accuracy: 73.74%
AUC-ROC:  0.6708

Detailed Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.44      0.37      0.40     15573
    Fake (1)       0.81      0.85      0.83     49808

    accuracy                           0.74     65381
   macro avg       0.63      0.61      0.62     65381
weighted avg       0.72      0.74      0.73     65381

